# Physics2Finance: Analysis Notebook

**Cross-Domain Latent Transfer from Physical Turbulence to Financial Market Microstructure**

This notebook walks through:
1. Turbulence structure function validation (confirming pre-training data quality)
2. LOB heatmap visualization and channel analogy verification
3. t-SNE/UMAP of physics latent space with LOB embeddings
4. Linear probe performance across horizons
5. Diebold-Mariano test summary table

In [ ]:
import sys
sys.path.insert(0, '..')

import numpy as np
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
%matplotlib inline
plt.rcParams['figure.dpi'] = 120
plt.rcParams['font.size'] = 11

## 1. Turbulence Structure Functions

Verify the pre-training data contains realistic multifractal turbulence.
Scaling exponents $\zeta(q)$ should deviate from the K41 linear prediction $q/3$.

In [ ]:
from src.data.fluid_dynamics.preprocessing import (
    compute_structure_function,
    estimate_scaling_exponents,
)

# Synthetic velocity field with intermittency for demonstration
# Replace with actual JHTDB data after running scripts/download_data.sh
np.random.seed(42)
H, W = 512, 512
x = np.linspace(0, 2*np.pi, W)
y = np.linspace(0, 2*np.pi, H)
X, Y = np.meshgrid(x, y)

# Synthetic turbulent-like field: sum of random Fourier modes
u = np.zeros((H, W))
for k in range(1, 20):
    phi = np.random.uniform(0, 2*np.pi)
    amp = k**(-5/6)  # K41 energy spectrum E(k) ~ k^{-5/3}
    u += amp * np.sin(k*X + phi) * np.cos(k*Y + np.random.uniform(0, 2*np.pi))

orders = [2, 4, 6]
r_vals, sq_vals = compute_structure_function(u, orders=orders, n_separations=25)
zeta = estimate_scaling_exponents(r_vals, sq_vals, inertial_range=(5, 80))

print("Scaling exponents ζ(q):")
for q, z in zip(orders, zeta):
    print(f"  q={q}: ζ(q) = {z:.3f}  (K41 = {q/3:.3f})")

In [ ]:
from src.utils.visualization import plot_structure_functions

fig = plot_structure_functions(
    r_vals, sq_vals, orders=orders,
    save_path='../outputs/evaluation/structure_functions.png'
)
plt.show()

## 2. LOB Heatmap Visualization

Demonstrate the RGB channel mapping:
- **R**: Ask depth (pressure)
- **G**: Bid depth (density)  
- **B**: Order imbalance (velocity)

In [ ]:
from src.data.financial.lob_to_heatmap import LOBHeatmapEncoder

# Simulate a LOB with a liquidity cascade event
np.random.seed(0)
T, levels = 100, 10
n_cols = 4 * levels

# Normal market conditions
window = np.random.exponential(10.0, (T, n_cols)).astype(np.float32)

# Inject a simulated liquidity shock at t=60 (bid side collapses)
window[60:, levels:2*levels] *= 0.1  # ask volume stays
window[60:, 3*levels:4*levels] *= 0.05  # bid volume collapses

enc = LOBHeatmapEncoder(levels=levels, img_size=224)
heatmap = enc.encode(window)

fig, axes = plt.subplots(1, 4, figsize=(16, 4))

rgb = np.transpose(heatmap, (1, 2, 0))
axes[0].imshow(rgb)
axes[0].set_title('RGB Composite\n(LOB Heatmap)')
axes[0].set_xlabel('Time →')
axes[0].set_ylabel('Price Level ↑')

channel_info = [
    ('Ask Depth\n(Pressure)', 'Reds'),
    ('Bid Depth\n(Density)', 'Greens'),
    ('Imbalance\n(Velocity)', 'Blues'),
]
for i, (title, cmap) in enumerate(channel_info):
    im = axes[i+1].imshow(heatmap[i], cmap=cmap, aspect='auto')
    axes[i+1].set_title(title)
    axes[i+1].axvline(x=int(224 * 0.6), color='red', linestyle='--', alpha=0.7, label='Shock')
    plt.colorbar(im, ax=axes[i+1], fraction=0.046)

plt.suptitle('LOB Heatmap: Liquidity Cascade at t=60\n(Analogous to Turbulent Enstrophy Burst)', 
             fontsize=12, fontweight='bold')
plt.tight_layout()
plt.show()

print("LOB Heatmap shape:", heatmap.shape)
print("Value range: [{:.3f}, {:.3f}]".format(heatmap.min(), heatmap.max()))

## 3. Physics Analogy: Side-by-Side Comparison

In [ ]:
from src.data.fluid_dynamics.preprocessing import field_to_rgb_image

# Synthetic turbulent pressure/velocity field for visual comparison
H, W = 224, 224
X, Y = np.meshgrid(np.linspace(0, 2*np.pi, W), np.linspace(0, 2*np.pi, H))
np.random.seed(7)

u_field = sum(np.random.randn() * k**(-5/6) * np.sin(k*X + np.random.uniform(0, 2*np.pi))
              for k in range(1, 15))
v_field = sum(np.random.randn() * k**(-5/6) * np.cos(k*Y + np.random.uniform(0, 2*np.pi))
              for k in range(1, 15))
p_field = sum(np.random.randn() * k**(-7/6) * np.sin(k*(X+Y) + np.random.uniform(0, 2*np.pi))
              for k in range(1, 10))

fluid_rgb = field_to_rgb_image(u_field, v_field, p_field, img_size=224)

from src.utils.visualization import plot_fluid_vs_lob
fig = plot_fluid_vs_lob(fluid_rgb, heatmap,
    titles=('Navier-Stokes DNS\n(Turbulent Field)', 'Limit Order Book\n(Liquidity Cascade)'))
plt.show()

## 4. Diebold-Mariano Test Demo

Simulated comparison to demonstrate the DM test infrastructure.

In [ ]:
from src.evaluation.diebold_mariano import diebold_mariano_test, run_full_dm_battery
from src.evaluation.metrics import evaluate_horizon

np.random.seed(42)
N = 2000

# Simulate ground-truth realized volatility (GARCH-like clustering)
rv_true = np.zeros(N)
rv_true[0] = 0.0001
for t in range(1, N):
    rv_true[t] = 0.00001 + 0.1 * np.random.randn()**2 + 0.85 * rv_true[t-1]
rv_true = np.abs(rv_true)

# PhyIP: slightly better than GARCH due to LOB information
phyip_pred = rv_true * np.exp(np.random.normal(0, 0.1, N))  # 10% noise

# GARCH: persistence model
garch_pred = np.roll(rv_true, 1)  # lagged RV
garch_pred[0] = rv_true.mean()
garch_pred = garch_pred * np.exp(np.random.normal(0, 0.25, N))  # 25% noise

# Random ViT: worse than both
rand_pred = rv_true.mean() + np.random.normal(0, rv_true.std() * 2, N)
rand_pred = np.abs(rand_pred)

models = {'PhyIP (Physics ViT + Linear Probe)': phyip_pred,
          'GARCH(1,1)': garch_pred,
          'Random ViT + Linear Probe': rand_pred}

print('=== Forecast Metrics ===')
for name, pred in models.items():
    m = evaluate_horizon(rv_true, pred)
    print(f"\n{name}:")
    print(f"  MAE  = {m['mae']:.6f}")
    print(f"  RMSE = {m['rmse']:.6f}")
    print(f"  QLIKE= {m['qlike']:.6f}")
    print(f"  R²   = {m['r2']:.4f}")

In [ ]:
print('\n=== Diebold-Mariano Tests (PhyIP vs. baselines) ===')

for name, pred in [('GARCH(1,1)', garch_pred), ('Random ViT', rand_pred)]:
    for loss_fn in ['mse', 'qlike']:
        result = diebold_mariano_test(rv_true, phyip_pred, pred, loss_fn=loss_fn)
        print(f"\nPhyIP vs. {name} [{loss_fn.upper()}]:")
        print(result)

## 5. Realized Volatility vs. Forecast (Visual)

Time-series overlay of RV and model predictions.

In [ ]:
from src.utils.visualization import plot_forecast_comparison

fig = plot_forecast_comparison(
    rv_true,
    {'PhyIP': phyip_pred, 'GARCH(1,1)': garch_pred, 'Random ViT': rand_pred},
    horizon=10,
    n_plot=400,
)
plt.show()

---

## Next Steps: Full Pipeline

1. **Download data**: `bash scripts/download_data.sh`
2. **Pre-train foundation model** *(compute-intensive — confirm before running)*:  
   `python -m src.training.pretrain_fluid --config configs/pretrain_config.yaml`
3. **Extract embeddings & fit linear probe**:  
   `python -m src.training.train_probe --config configs/probe_config.yaml`
4. **Full evaluation**:  
   `bash scripts/run_evaluation.sh`